# Step 1. Data Loading (Source: https://github.com/rakshitha123/TSForecasting/tree/master/utils)

In [1]:
from datetime import datetime
from distutils.util import strtobool

import pandas as pd
import numpy as np

# Converts the contents in a .tsf file into a dataframe and returns it along with other meta-data of the dataset: frequency, horizon, whether the dataset contains missing values and whether the series have equal lengths
#
# Parameters
# full_file_path_and_name - complete .tsf file path
# replace_missing_vals_with - a term to indicate the missing values in series in the returning dataframe
# value_column_name - Any name that is preferred to have as the name of the column containing series values in the returning dataframe
def convert_tsf_to_dataframe(
    full_file_path_and_name,
    replace_missing_vals_with="NaN",
    value_column_name="series_value",
):
    col_names = []
    col_types = []
    all_data = {}
    line_count = 0
    frequency = None
    forecast_horizon = None
    contain_missing_values = None
    contain_equal_length = None
    found_data_tag = False
    found_data_section = False
    started_reading_data_section = False

    with open(full_file_path_and_name, "r", encoding="cp1252") as file:
        for line in file:
            # Strip white space from start/end of line
            line = line.strip()

            if line:
                if line.startswith("@"):  # Read meta-data
                    if not line.startswith("@data"):
                        line_content = line.split(" ")
                        if line.startswith("@attribute"):
                            if (
                                len(line_content) != 3
                            ):  # Attributes have both name and type
                                raise Exception("Invalid meta-data specification.")

                            col_names.append(line_content[1])
                            col_types.append(line_content[2])
                        else:
                            if (
                                len(line_content) != 2
                            ):  # Other meta-data have only values
                                raise Exception("Invalid meta-data specification.")

                            if line.startswith("@frequency"):
                                frequency = line_content[1]
                            elif line.startswith("@horizon"):
                                forecast_horizon = int(line_content[1])
                            elif line.startswith("@missing"):
                                contain_missing_values = bool(
                                    strtobool(line_content[1])
                                )
                            elif line.startswith("@equallength"):
                                contain_equal_length = bool(strtobool(line_content[1]))

                    else:
                        if len(col_names) == 0:
                            raise Exception(
                                "Missing attribute section. Attribute section must come before data."
                            )

                        found_data_tag = True
                elif not line.startswith("#"):
                    if len(col_names) == 0:
                        raise Exception(
                            "Missing attribute section. Attribute section must come before data."
                        )
                    elif not found_data_tag:
                        raise Exception("Missing @data tag.")
                    else:
                        if not started_reading_data_section:
                            started_reading_data_section = True
                            found_data_section = True
                            all_series = []

                            for col in col_names:
                                all_data[col] = []

                        full_info = line.split(":")

                        if len(full_info) != (len(col_names) + 1):
                            raise Exception("Missing attributes/values in series.")

                        series = full_info[len(full_info) - 1]
                        series = series.split(",")

                        if len(series) == 0:
                            raise Exception(
                                "A given series should contains a set of comma separated numeric values. At least one numeric value should be there in a series. Missing values should be indicated with ? symbol"
                            )

                        numeric_series = []

                        for val in series:
                            if val == "?":
                                numeric_series.append(replace_missing_vals_with)
                            else:
                                numeric_series.append(float(val))

                        if numeric_series.count(replace_missing_vals_with) == len(
                            numeric_series
                        ):
                            raise Exception(
                                "All series values are missing. A given series should contains a set of comma separated numeric values. At least one numeric value should be there in a series."
                            )

                        all_series.append(pd.Series(numeric_series).array)

                        for i in range(len(col_names)):
                            att_val = None
                            if col_types[i] == "numeric":
                                att_val = int(full_info[i])
                            elif col_types[i] == "string":
                                att_val = str(full_info[i])
                            elif col_types[i] == "date":
                                att_val = datetime.strptime(
                                    full_info[i], "%Y-%m-%d %H-%M-%S"
                                )
                            else:
                                raise Exception(
                                    "Invalid attribute type."
                                )  # Currently, the code supports only numeric, string and date types. Extend this as required.

                            if att_val is None:
                                raise Exception("Invalid attribute value.")
                            else:
                                all_data[col_names[i]].append(att_val)

                line_count = line_count + 1

        if line_count == 0:
            raise Exception("Empty file.")
        if len(col_names) == 0:
            raise Exception("Missing attribute section.")
        if not found_data_section:
            raise Exception("Missing series information under data section.")

        all_data[value_column_name] = all_series
        loaded_data = pd.DataFrame(all_data)

        return (
            loaded_data,
            frequency,
            forecast_horizon,
            contain_missing_values,
            contain_equal_length,
        )


# Example of usage
loaded_data, frequency, forecast_horizon, contain_missing_values, contain_equal_length = convert_tsf_to_dataframe("australian_electricity_demand_dataset.tsf")

print(loaded_data)
print(frequency)
print(forecast_horizon)
print(contain_missing_values)
print(contain_equal_length)

  series_name state start_timestamp  \
0          T1   NSW      2002-01-01   
1          T2   VIC      2002-01-01   
2          T3   QUN      2002-01-01   
3          T4    SA      2002-01-01   
4          T5   TAS      2002-01-01   

                                        series_value  
0  [5714.045004, 5360.189078, 5014.835118, 4602.7...  
1  [3535.867064, 3383.499028, 3655.527552, 3510.4...  
2  [3382.041342, 3288.315794, 3172.329022, 3020.3...  
3  [1191.078014, 1219.589472, 1119.173498, 1016.4...  
4  [315.915504, 306.245864, 305.762576, 295.60219...  
half_hourly
None
False
False


# Step 2. Data Preprocessing

In [4]:
# 1. Initialize lists to accumulate data for all rolling windows
all_contexts = []
all_horizons = []

# 2. Loop through all 5 series in the loaded dataframe
for idx in range(1):
    series_row = loaded_data.iloc[3]
    state_id = series_row["state"]
    start_date = series_row["start_timestamp"]
    series_values = pd.Series(series_row["series_value"])

    # Reconstruct the full hourly DatetimeIndex and downsample to hourly
    full_index = pd.date_range(start=start_date, periods=len(series_values), freq="30min")
    series_values.index = full_index
    hourly_series = series_values.resample("h").mean()

    # 3. Implement the rolling window over 365 days of 2014
    # To get 168 hours of context before 2014-01-01 00:00, start 7 days prior (2013-12-25 00:00)
    start_anchor = pd.Timestamp("2013-12-25 00:00:00")

    for day in range(365):
        # Calculate sliding windows stepping by 24 hours at each iteration
        window_start = start_anchor + pd.Timedelta(hours=day * 24)
        context_end = window_start + pd.Timedelta(hours=168)
        horizon_end = context_end + pd.Timedelta(hours=24)

        # Extract context (168h)
        context_part = hourly_series.loc[window_start : context_end - pd.Timedelta(hours=1)].reset_index()
        context_part.columns = ["timestamp", "target"]

        # Create unique ID per rolling window (e.g., "AUS_NSW_day_0") to keep them separate in Chronos-2
        context_part["id"] = f"AUS_{state_id}_day_{day}"

        # Extract ground truth horizon (24h)
        horizon_part = hourly_series.loc[context_end : horizon_end - pd.Timedelta(hours=1)].values

        all_contexts.append(context_part)
        all_horizons.append(horizon_part)

In [5]:
# 3. Combine all series context into a single long-format DataFrame
context_df = pd.concat(all_contexts, ignore_index=True)

# 4. Stack horizons into a 2D array of shape (365, 24)
horizon_true = np.array(all_horizons)

# 5. Check missing or infinite values
assert not context_df["target"].isna().any(), "Context contains NaN values!"

# 6. Save outputs
context_df.to_csv("context_df.csv", index=False)
np.save("horizon_true.npy", horizon_true)

# 7. Display previews of the generated outputs
print("--- Context DataFrame Preview (First & Last 5 Rows) ---")
display(context_df)

print("\n--- Ground Truth (horizon_true) Array Preview ---")
print(f"Array Shape: {horizon_true.shape}")
print(horizon_true)

--- Context DataFrame Preview (First & Last 5 Rows) ---


,timestamp,target,id
0,2013-12-25 00:00:00,1099.782098,AUS_SA_day_0
1,2013-12-25 01:00:00,931.492157,AUS_SA_day_0
2,2013-12-25 02:00:00,839.934374,AUS_SA_day_0
3,2013-12-25 03:00:00,797.504308,AUS_SA_day_0
4,2013-12-25 04:00:00,791.820920,AUS_SA_day_0
...,...,...,...
61315,2014-12-30 19:00:00,1113.653309,AUS_SA_day_364
61316,2014-12-30 20:00:00,1133.053521,AUS_SA_day_364
61317,2014-12-30 21:00:00,1090.616834,AUS_SA_day_364
61318,2014-12-30 22:00:00,1019.414251,AUS_SA_day_364



--- Ground Truth (horizon_true) Array Preview ---
Array Shape: (365, 24)
[[1401.556675 1252.007041 1169.560279 ... 1232.828455 1139.467277
  1194.483795]
 [1157.267032  993.426996  929.497223 ... 1095.73524  1028.966713
  1119.487377]
 [1126.067836  949.898729  861.594444 ... 1067.291173 1009.437787
  1096.129751]
 ...
 [1064.741557  931.276595  859.879672 ... 1075.612713 1017.446622
  1102.063325]
 [1092.640927  953.792129  875.165409 ... 1090.616834 1019.414251
  1108.118839]
 [1101.331609  941.086216  855.310916 ... 1056.919743 1039.711744
  1155.492353]]
